# 🔬 Advanced Analytics — Mutual Fund Analytics
**Capstone Project I | Bluestock Fintech Internship**  
**Author:** Tejaswini | B.Tech AI & ML, KITS Karimnagar  
**Date:** July 2026  

---

## Overview
This notebook covers advanced quantitative analytics:
Historical VaR & CVaR · Rolling Sharpe · Cohort Analysis · SIP Continuity · Fund Recommender · Sector HHI


## 0. Setup

In [ ]:
import os, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import Image, display

warnings.filterwarnings('ignore')
%matplotlib inline

plt.rcParams.update({
    'figure.facecolor':'#0f1117','axes.facecolor':'#1a1d2e',
    'axes.edgecolor':'#3a3d5c','axes.labelcolor':'#e0e0e0',
    'xtick.color':'#a0a0b0','ytick.color':'#a0a0b0',
    'text.color':'#e0e0e0','grid.color':'#2a2d4e',
    'grid.linestyle':'--','grid.alpha':0.4,'font.size':10,
})

RAW    = r'C:\Users\tejas\OneDrive\Documents\Mutualfundanalysis\data\raw'
CHARTS = r'C:\Users\tejas\OneDrive\Documents\Mutualfundanalysis\reports\charts'
REP    = r'C:\Users\tejas\OneDrive\Documents\Mutualfundanalysis\reports'
RF_DAILY = 0.065 / 252
print('Setup complete ✅')

## 1. Load Data

In [ ]:
nav   = pd.read_csv(f'{RAW}/02_nav_history.csv', parse_dates=['date'])
fund  = pd.read_csv(f'{RAW}/01_fund_master.csv')
trans = pd.read_csv(f'{RAW}/08_investor_transactions.csv', parse_dates=['transaction_date'])
hold  = pd.read_csv(f'{RAW}/09_portfolio_holdings.csv')
perf  = pd.read_csv(f'{RAW}/07_scheme_performance.csv')
nav   = nav.merge(fund[['amfi_code','scheme_name','sub_category','plan','risk_category','fund_house']], on='amfi_code', how='left')
nav.sort_values(['amfi_code','date'], inplace=True)
nav['daily_return'] = nav.groupby('amfi_code')['nav'].pct_change()
nav_clean = nav.dropna(subset=['daily_return']).copy()
print(f'Loaded {len(nav_clean):,} NAV records ✅')

## 2. Historical VaR (95%) & CVaR


**VaR (Value at Risk):** The maximum expected loss at a given confidence level.

**Formula:** `VaR_95 = 5th percentile of daily returns`

**CVaR (Conditional VaR / Expected Shortfall):** Mean of all returns below VaR — tells you how bad the bad days really are.

**Formula:** `CVaR = mean(returns where return ≤ VaR)`

**Insight 1:** Small Cap funds have the highest VaR (-2.7% daily) meaning on a bad day you could lose 2.7% — nearly 3× more than Large Cap funds (-0.9%). CVaR for Small Cap reaches -3.2%, showing extreme tail risk.


In [ ]:
var_df = pd.read_csv(f'{REP}/var_cvar_report.csv')
display(Image(f'{CHARTS}/var_cvar_chart.png', width=900))
print(var_df[['scheme_name','var_95_daily_pct','cvar_95_daily_pct','sub_category']].head(10).to_string(index=False))

## 3. Rolling 90-Day Sharpe Ratio


**Formula:** `Sharpe = (rolling_mean(return) - Rf) / rolling_std(return) × √252`

Rolling window = 90 trading days (~4 months). Shows how risk-adjusted performance evolves over time.

**Insight 2:** Rolling Sharpe reveals that even top-performing funds experienced negative Sharpe during the 2024 correction period, confirming the market-wide risk-off sentiment was temporary.


In [ ]:
display(Image(f'{CHARTS}/rolling_sharpe_chart.png', width=900))

## 4. Investor Cohort Analysis


Investors grouped by **year of first transaction**. Shows whether newer cohorts invest more or less than older ones.

**Insight 3:** The 2025 cohort shows 22.8% higher average SIP amount (₹13,505) vs 2024 cohort (₹10,997), suggesting newer investors are more financially aware and confident in committing larger SIP amounts.


In [ ]:
cohort = pd.read_csv(f'{REP}/cohort_analysis.csv')
display(Image(f'{CHARTS}/cohort_analysis_chart.png', width=900))
print(cohort.to_string(index=False))

## 5. SIP Continuity Analysis


For investors with 6+ SIP transactions, compute average gap between SIP dates.

**At-Risk flag:** `avg gap > 35 days` (monthly SIPs should be ~30 days)

**Insight 4:** Only 2.2% of SIP investors maintain regular monthly cadence. 97.8% are flagged At-Risk with avg gaps exceeding 35 days — indicating irregular SIP behaviour possibly due to mandate failures or voluntary pauses.


In [ ]:
sip_cont = pd.read_csv(f'{REP}/sip_continuity.csv')
display(Image(f'{CHARTS}/sip_continuity_chart.png', width=900))
print(f'Regular: {len(sip_cont[sip_cont["status"]=="Regular"]):,}')
print(f'At-Risk: {len(sip_cont[sip_cont["status"]=="At-Risk"]):,}')

## 6. Fund Recommender


**Logic:** Input investor risk appetite → filter matching `risk_category` funds → rank by Sharpe Ratio → return top 3.

Risk mapping: Low → Liquid/Debt funds | Moderate → Large Cap | High → Mid/Small Cap


In [ ]:
import sys
sys.path.insert(0, r'C:\Users\tejas\OneDrive\Documents\Mutualfundanalysis')
from recommender import load_data, recommend_funds, print_recommendation
fund_data, perf_data = load_data()
for appetite in ['Low', 'Moderate', 'High']:
    result = recommend_funds(appetite, fund_data, perf_data)
    print_recommendation(appetite, result)

## 7. Sector HHI Concentration


**HHI (Herfindahl-Hirschman Index):** `Σ(weight_i / 100)²` per fund

HHI > 0.15 = Highly concentrated | 0.10–0.15 = Moderate | < 0.10 = Diversified

**Insight 5:** Most equity funds show moderate HHI (0.10–0.15), suggesting reasonable sector diversification. Funds with Banking as top sector show higher concentration risk given India's market-cap-heavy financial sector.


In [ ]:
hhi_df = pd.read_csv(f'{REP}/hhi_concentration.csv')
display(Image(f'{CHARTS}/hhi_concentration_chart.png', width=900))
print(hhi_df[['scheme_name','hhi_score','concentration','top_sector','num_stocks']].to_string(index=False))

---

## 8. Advanced Insights Summary

| # | Insight |
|---|--------|
| 1 | Small Cap funds have highest daily VaR (-2.7%) — 3× riskier than Large Cap on bad days |
| 2 | Rolling Sharpe dips below 0 for all funds during 2024 correction — market-wide, not fund-specific |
| 3 | 2025 investor cohort invests 22.8% more per SIP than 2024 cohort — rising financial confidence |
| 4 | Only 2.2% of investors maintain regular SIP cadence — 97.8% are at-risk of discontinuity |
| 5 | Most equity funds are moderately concentrated — Banking dominates across all equity portfolios |
